# Revisión de las etiquetas de RoCoLe

Queremos entender las fotos de RoCoLe, sus etiquetas y sus límites, antes de armar características para clasificar lo que se ve en la hoja. RoCoLe es el conjunto principal. BRACOL se mira aparte: cambian la especie, la captura y las etiquetas.


In [ ]:
import hashlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import shutil
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split


In [ ]:
rocole_classes_path = Path("../../data/raw/RoCoLe/Annotations/RoCoLe-classes.xlsx")
rocole_classes_path


In [ ]:
rocole_classes = pd.read_excel(rocole_classes_path)
rocole_classes.head(5)


File es el nombre de la foto. Binary.Label dice si está sana o afectada. Multiclass.Label afina la afectada: ácaro, o un nivel de la misma roya.


Primero contamos las etiquetas y revisamos si cada foto abre y si hay copias idénticas. La roya va del nivel 1 (1–5 % del área) al 4 (más del 50 %). Las categorías son sana, ácaro y esos cuatro niveles. Todavía no agrupamos la roya. Las copias y, si se puede, las fotos de la misma planta quedan en el mismo lado. El nombre del archivo no se usa para predecir.

Después miramos las fotos por categoría, separamos la hoja del fondo y recién ahí armamos características de color, índices RGB, textura y zonas sospechosas. La métrica prevista es el F1 macro. BRACOL se describe aparte y no elige el modelo de RoCoLe.


Contamos filas y columnas de la tabla de etiquetas.


In [ ]:
# rocole_classes, ya cargada.

rocole_class_shape = rocole_classes.shape

rocole_class_shape


Contamos cuántas fotos hay en cada etiqueta binaria.


In [ ]:
# rocole_classes

binary_observed = rocole_classes["Binary.Label"].value_counts(dropna=False)
binary_label_order = ["healthy", "unhealthy"]
binary_label_order = [
    label for label in binary_label_order if label in binary_observed.index
]
for label in binary_observed.index:
    if label not in binary_label_order:
        binary_label_order.append(label)

photo_count = binary_observed.loc[binary_label_order].astype(int)
binary_label_counts = pd.DataFrame(
    {
        "photo_count": photo_count,
        "percent": (photo_count / photo_count.sum() * 100).round(1),
    }
)
binary_label_counts.index.name = "Binary.Label"

binary_label_counts


Contamos la multiclase. Los niveles de roya van por separado.


In [ ]:
# rocole_classes

multiclass_observed = rocole_classes["Multiclass.Label"].value_counts(dropna=False)
multiclass_label_order = [
    "healthy",
    "red_spider_mite",
    "rust_level_1",
    "rust_level_2",
    "rust_level_3",
    "rust_level_4",
]
multiclass_label_order = [
    label for label in multiclass_label_order if label in multiclass_observed.index
]
for label in multiclass_observed.index:
    if label not in multiclass_label_order:
        multiclass_label_order.append(label)

photo_count = multiclass_observed.loc[multiclass_label_order].astype(int)
multiclass_label_counts = pd.DataFrame(
    {
        "photo_count": photo_count,
        "percent": (photo_count / photo_count.sum() * 100).round(1),
    }
)
multiclass_label_counts.index.name = "Multiclass.Label"

multiclass_label_counts


Preparamos el gráfico de barras.


In [ ]:
# No hay variables previas en esta celda.


# Queda disponible plt para el gráfico.


Dibujamos una barra por categoría, con el conteo encima. El nombre en español es solo del gráfico.


In [ ]:
# multiclass_label_counts

multiclass_display_names = {
    "healthy": "Sana",
    "red_spider_mite": "Ácaro",
    "rust_level_1": "Roya nivel 1",
    "rust_level_2": "Roya nivel 2",
    "rust_level_3": "Roya nivel 3",
    "rust_level_4": "Roya nivel 4",
}
bar_labels = [
    multiclass_display_names.get(str(label), str(label))
    for label in multiclass_label_counts.index
]
photo_counts = multiclass_label_counts["photo_count"].astype(int)

figure, axis = plt.subplots(figsize=(10, 4.5))
bars = axis.bar(bar_labels, photo_counts, color="#3d6b4f")
axis.bar_label(bars, padding=3)
axis.set_ylim(0, photo_counts.max() * 1.15)
axis.set_title("Fotografías de RoCoLe por categoría")
axis.set_xlabel("Categoría")
axis.set_ylabel("Número de fotografías")
axis.tick_params(axis="x", labelrotation=20)
figure.tight_layout()
figure_path = Path("../../experiments/results/rocole-multiclass-label-counts.png")
figure.savefig(figure_path, dpi=120, bbox_inches="tight")


Hay 1.560 fotos y 3 columnas. En binario están casi parejas: 791 sanas (50,7 %) y 769 afectadas (49,3 %). En multiclase el desbalance sí aparece: sana sigue en 791, roya nivel 1 tiene 344 (22,1 %), ácaro 167 (10,7 %), roya nivel 2 tiene 166 (10,6 %), nivel 3 tiene 62 (4,0 %) y nivel 4 tiene 30 (1,9 %). Son las filas del Excel, antes de mirar duplicados.


Vemos qué parte del total ocupa cada etiqueta. A la izquierda va la binaria y a la derecha la multiclase.


In [ ]:
# binary_label_counts, multiclass_label_counts

# Nombres solo para el gráfico. Las tablas de conteo no se traducen.
binary_display_names = {
    "healthy": "Sana",
    "unhealthy": "Afectada",
}
multiclass_display_names = {
    "healthy": "Sana",
    "red_spider_mite": "Ácaro",
    "rust_level_1": "Roya nivel 1",
    "rust_level_2": "Roya nivel 2",
    "rust_level_3": "Roya nivel 3",
    "rust_level_4": "Roya nivel 4",
}
pie_colors = ["#3d6b4f", "#c4a35a", "#d4a017", "#c47b2b", "#a85a32", "#7a3b2e"]

binary_names = [
    binary_display_names.get(str(label), str(label))
    for label in binary_label_counts.index
]
binary_photo_counts = binary_label_counts["photo_count"].astype(int)

multiclass_names = [
    multiclass_display_names.get(str(label), str(label))
    for label in multiclass_label_counts.index
]
multiclass_photo_counts = multiclass_label_counts["photo_count"].astype(int)

figure, axes = plt.subplots(1, 2, figsize=(12, 5.4))
axes[0].pie(
    binary_photo_counts,
    labels=binary_names,
    autopct="%1.1f%%",
    startangle=90,
    colors=pie_colors[: len(binary_names)],
)
axes[0].set_title("Etiqueta binaria")

wedges, _ = axes[1].pie(
    multiclass_photo_counts,
    startangle=90,
    colors=pie_colors[: len(multiclass_names)],
)
axes[1].set_title("Etiqueta multiclase")
legend_labels = [
    f"{name}  {percent:.1f} %"
    for name, percent in zip(multiclass_names, multiclass_label_counts["percent"])
]
axes[1].legend(
    wedges,
    legend_labels,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    frameon=False,
)
figure.suptitle("RoCoLe: proporción de cada etiqueta", y=1.02)
figure.tight_layout()

figure_path = Path("../../experiments/results/rocole-label-pies.png")
figure.savefig(figure_path, dpi=120, bbox_inches="tight")


La binaria queda casi partida en dos. En la multiclase, sana es la mitad, sigue roya nivel 1, ácaro y roya nivel 2 se parecen, y los niveles 3 y 4 son los trozos más finos. Son los mismos porcentajes de las tablas.


Seguimos con RoCoLe: cada etiqueta debería tener su foto, sin duplicados y con imágenes que abran. Los archivos se quedan. Aquí queda la carpeta.


In [ ]:
# Path

rocole_photos_dir = Path("../../data/raw/RoCoLe/Photos")

rocole_photos_dir


Miramos si falta el nombre de la foto o alguna etiqueta.


In [ ]:
# rocole_classes

missing_value_counts = rocole_classes.isna().sum()

missing_value_counts


Miramos si el mismo nombre aparece más de una vez en la tabla.


In [ ]:
# rocole_classes

file_name_counts = rocole_classes["File"].value_counts()
repeated_file_names = file_name_counts[file_name_counts > 1]

repeated_file_names


Miramos si una fila dice sana en una columna y otra cosa en la otra.


In [ ]:
# rocole_classes

# healthy debe ir con healthy. unhealthy debe ir con otra categoría.
binary_says_healthy = rocole_classes["Binary.Label"] == "healthy"
multiclass_says_healthy = rocole_classes["Multiclass.Label"] == "healthy"
label_mismatches = rocole_classes[binary_says_healthy != multiclass_says_healthy]

label_mismatches


Miramos qué nombres del Excel no tienen un jpg en la carpeta.


In [ ]:
# rocole_classes, rocole_photos_dir

photo_names = {path.name for path in rocole_photos_dir.glob("*.jpg")}
label_names = set(rocole_classes["File"])
missing_photos = sorted(label_names - photo_names)

missing_photos


Miramos qué jpg de la carpeta no están en el Excel.


In [ ]:
# photo_names, label_names

extra_photos = sorted(photo_names - label_names)

extra_photos


Preparamos la lectura de cada foto.


In [ ]:
# No hay variables de datos en esta celda.


# Queda disponible Image.


Listamos los archivos que no abren.


In [ ]:
# rocole_classes, rocole_photos_dir, Image

unreadable_photos = []
for file_name in rocole_classes["File"]:
    photo_path = rocole_photos_dir / file_name
    try:
        with Image.open(photo_path) as photo:
            photo.load()
    except Exception:
        unreadable_photos.append(file_name)

unreadable_photos


Comparamos el contenido de los archivos. Si dos jpg son idénticos, quedan en el mismo grupo aunque el nombre cambie.


In [ ]:
# rocole_classes, rocole_photos_dir


photo_hash_groups = {}
for file_name in rocole_classes["File"]:
    photo_bytes = (rocole_photos_dir / file_name).read_bytes()
    photo_hash = hashlib.md5(photo_bytes).hexdigest()
    photo_hash_groups.setdefault(photo_hash, []).append(file_name)

duplicate_photo_groups = [
    names for names in photo_hash_groups.values() if len(names) > 1
]
duplicate_group_count = len(duplicate_photo_groups)
files_per_group = {len(names) for names in duplicate_photo_groups}

duplicate_group_count, files_per_group


Juntamos cada copia con la etiqueta de su fila. El nombre sirve para unir.


In [ ]:
# duplicate_photo_groups, rocole_classes

duplicate_name_rows = []
for group_number, names in enumerate(duplicate_photo_groups, start=1):
    for file_name in names:
        duplicate_name_rows.append({"group": group_number, "File": file_name})

duplicate_name_table = pd.DataFrame(duplicate_name_rows)
duplicate_photo_labels = duplicate_name_table.merge(rocole_classes, on="File")

duplicate_photo_labels


Contamos los pares que coinciden y los que no.


In [ ]:
# duplicate_photo_labels

same_labels = 0
different_multiclass_only = 0
different_binary = 0
for _, group in duplicate_photo_labels.groupby("group"):
    same_binary = group["Binary.Label"].nunique() == 1
    same_multiclass = group["Multiclass.Label"].nunique() == 1
    if not same_binary:
        different_binary += 1
    elif not same_multiclass:
        different_multiclass_only += 1
    else:
        same_labels += 1

duplicate_label_summary = pd.Series(
    {
        "same_labels": same_labels,
        "different_multiclass_only": different_multiclass_only,
        "different_binary": different_binary,
    },
    name="group_count",
)

duplicate_label_summary


Dejamos escritas las copias que no comparten etiqueta.


In [ ]:
# duplicate_photo_labels

conflict_group_numbers = []
for group_number, group in duplicate_photo_labels.groupby("group"):
    same_binary = group["Binary.Label"].nunique() == 1
    same_multiclass = group["Multiclass.Label"].nunique() == 1
    if not (same_binary and same_multiclass):
        conflict_group_numbers.append(group_number)

duplicate_label_conflicts = duplicate_photo_labels[
    duplicate_photo_labels["group"].isin(conflict_group_numbers)
]
pd.set_option("display.max_rows", 200)

duplicate_label_conflicts


La tabla está completa: no falta ningún nombre ni etiqueta, ningún nombre se repite y, en cada fila, sana va con sana. Las 1.560 fotos existen y abren.

El problema son 67 pares de fotos idénticas con distinto nombre (134 archivos). De ellos, 40 tienen la misma etiqueta binaria y multiclase: esos se tratan a continuación. Quedan pendientes los 27 con conflicto (26 con multiclase distinta y 1 con binaria distinta: C11P23E1.jpg como ácaro y C12P36E1.jpg como sana). Cuando se arme la partición, las copias van al mismo lado. Esta revisión no mira fotos parecidas ni agrupa por planta.


Ampliamos la revisión solo para los 40 pares idénticos con la misma etiqueta: dejamos una imagen por par y anotamos cuál se retira, sin tocar aún los pares con etiqueta distinta.


De los pares de fotos idénticas, nos quedamos solo con los que comparten etiqueta binaria y multiclase; son los 40 donde no hay conflicto de etiqueta.


In [ ]:
# duplicate_photo_labels

same_label_group_numbers = []
for group_number, group in duplicate_photo_labels.groupby("group"):
    same_binary = group["Binary.Label"].nunique() == 1
    same_multiclass = group["Multiclass.Label"].nunique() == 1
    if same_binary and same_multiclass:
        same_label_group_numbers.append(group_number)

same_label_duplicate_labels = duplicate_photo_labels[
    duplicate_photo_labels["group"].isin(same_label_group_numbers)
].copy()

same_label_duplicate_labels


En cada uno de esos pares hay que decidir una sola imagen a conservar. Usamos el nombre de archivo que va primero en orden alfabético, para que la elección sea estable y se pueda repetir.


In [ ]:
# same_label_duplicate_labels

same_label_kept_files = (
    same_label_duplicate_labels.groupby("group", as_index=False)["File"]
    .min()
    .rename(columns={"File": "kept_file"})
)

same_label_kept_files


La otra copia del par no se borra del disco; solo se anota para no usarla después, con la razón de que es duplicado exacto y misma etiqueta.


In [ ]:
# same_label_duplicate_labels, same_label_kept_files

same_label_withdrawals = same_label_duplicate_labels.merge(
    same_label_kept_files, on="group"
)
same_label_withdrawals = same_label_withdrawals[
    same_label_withdrawals["File"] != same_label_withdrawals["kept_file"]
][["group", "File", "kept_file"]].rename(columns={"File": "withdrawn_file"})
same_label_withdrawals["reason"] = "exact_duplicate_same_label"
same_label_withdrawals = same_label_withdrawals.reset_index(drop=True)

same_label_withdrawals


En la tabla de etiquetas de trabajo quitamos las filas de las copias retiradas, para que cada foto idéntica con etiqueta igual aparezca una sola vez.


In [ ]:
# rocole_classes, same_label_withdrawals

withdrawn_files = set(same_label_withdrawals["withdrawn_file"])
rocole_classes_same_label_deduped = rocole_classes[
    ~rocole_classes["File"].isin(withdrawn_files)
].copy()

rocole_classes_same_label_deduped


Guardamos en curated el registro de retiradas, para que el equipo vea qué se apartó y por qué sin volver a correr el cuaderno.


In [ ]:
# same_label_withdrawals

same_label_withdrawals_path = Path(
    "../../data/curated/rocole/withdrawals/same_label_duplicate_withdrawals.csv"
)
same_label_withdrawals_path.parent.mkdir(parents=True, exist_ok=True)
same_label_withdrawals.to_csv(same_label_withdrawals_path, index=False)

same_label_withdrawals_path


También dejamos en curated la tabla de etiquetas de trabajo, ya sin las 40 copias retiradas.


In [ ]:
# rocole_classes_same_label_deduped

rocole_classes_curated_path = Path("../../data/curated/rocole/classes.csv")
rocole_classes_curated_path.parent.mkdir(parents=True, exist_ok=True)
rocole_classes_same_label_deduped.to_csv(rocole_classes_curated_path, index=False)

rocole_classes_curated_path


Con esto, los 40 pares quedan en una sola fila útil cada uno; se retiraron 40 copias del trabajo (sin borrarlas del disco) y la tabla en curated baja a 1.520 filas. Los 27 pares con etiqueta distinta siguen pendientes. featured queda para características y el split.


Ya resolvimos los 40 pares con la misma etiqueta. De los que quedan con conflicto, primero miramos solo los que discuten el nivel de roya: los revisamos en el cuaderno con un mosaico y ayuda de IA generativa, no con un PDF. Todavía no entran los de ácaro ni el par sana/ácaro.


Nos quedamos solo con los pares idénticos donde las dos etiquetas multiclase son niveles de roya distintos.


In [ ]:
# duplicate_photo_labels

rust_severity_conflict_group_numbers = []
for group_number, group in duplicate_photo_labels.groupby("group"):
    multiclass_labels = group["Multiclass.Label"]
    both_rust_levels = multiclass_labels.str.startswith("rust_level_").all()
    distinct_levels = multiclass_labels.nunique() > 1
    if both_rust_levels and distinct_levels:
        rust_severity_conflict_group_numbers.append(group_number)

rust_severity_conflict_labels = duplicate_photo_labels[
    duplicate_photo_labels["group"].isin(rust_severity_conflict_group_numbers)
].copy()

rust_severity_conflict_labels


Contamos cuántos grupos son: deberían ser 15.


In [ ]:
# rust_severity_conflict_labels

rust_severity_conflict_group_count = rust_severity_conflict_labels["group"].nunique()

rust_severity_conflict_group_count


Seguimos con los 15 pares de solo severidad de roya. Los miramos en un mosaico y el análisis de qué etiqueta conservar lo hacemos con Astra (Claude Opus). Después dejamos el rastro en curated. Todavía no entran los 11 ácaro/roya ni el par sana/ácaro.


In [ ]:
# rust_severity_conflict_labels

# Una fila por grupo: columnas de decisión vacías; el método ya es IA generativa.
decision_rows = []
sorted_group_numbers = sorted(rust_severity_conflict_labels["group"].unique())
for case_number, group_number in enumerate(sorted_group_numbers, start=1):
    group_rows = (
        rust_severity_conflict_labels[
            rust_severity_conflict_labels["group"] == group_number
        ]
        .sort_values("File")
        .reset_index(drop=True)
    )
    decision_rows.append(
        {
            "case": case_number,
            "group": group_number,
            "file_a": group_rows.loc[0, "File"],
            "label_a": group_rows.loc[0, "Multiclass.Label"],
            "file_b": group_rows.loc[1, "File"],
            "label_b": group_rows.loc[1, "Multiclass.Label"],
            "chosen_label": "",
            "kept_file": "",
            "withdrawn_file": "",
            "decision_method": "generative_ai",
            "notes": "",
        }
    )

rust_severity_review_decisions = pd.DataFrame(decision_rows)

rust_severity_review_decisions


Mostramos los 15 pares en un mosaico dentro del cuaderno: cada fila es un caso, con las dos fotos lado a lado y debajo el nombre de archivo y su etiqueta multiclase.


In [ ]:
# rust_severity_conflict_labels, rocole_photos_dir, Image


sorted_group_numbers = sorted(rust_severity_conflict_labels["group"].unique())
case_count = len(sorted_group_numbers)
fig, axes = plt.subplots(case_count, 2, figsize=(14, 3.2 * case_count))

for row_index, group_number in enumerate(sorted_group_numbers):
    group_rows = (
        rust_severity_conflict_labels[
            rust_severity_conflict_labels["group"] == group_number
        ]
        .sort_values("File")
        .reset_index(drop=True)
    )
    for column_index in range(2):
        ax = axes[row_index, column_index]
        row = group_rows.iloc[column_index]
        photo_path = rocole_photos_dir / row["File"]
        with Image.open(photo_path) as photo:
            ax.imshow(photo.convert("RGB"))
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel(
            f"{row['File']}\n{row['Multiclass.Label']}",
            fontsize=8,
        )
        if column_index == 0:
            ax.set_ylabel(f"Caso {row_index + 1}", fontsize=9)

fig.suptitle("Pares idénticos con niveles de roya distintos", fontsize=14)
fig.tight_layout()
plt.show()

# Mosaico mostrado en el cuaderno; no se escribe PDF.


El mosaico de arriba (15 filas, dos fotos idénticas por fila) se envía a Astra para decidir, caso a caso, qué nivel de roya conservar. Abajo queda el prompt usado. Cuando Astra responda, pegamos el CSV de decisiones y actualizamos curated.


Prompt enviado a Astra (Claude Opus) junto con el mosaico completo:

```
Eres un revisor visual de severidad de roya del café (Hemileia vastatrix) sobre fotos del dataset RoCoLe.

Te adjunto UN mosaico con 15 filas y 2 columnas. Cada fila es un caso: las dos fotos de la fila son el mismo archivo duplicado (contenido idéntico) con distinta etiqueta de severidad. Revisa las 15 filas de arriba a abajo (caso 1 = primera fila … caso 15 = última fila).

Escala RoCoLe (por % aproximado del área de la hoja afectada por roya):
- rust_level_1: 1–5 %
- rust_level_2: 6–20 %
- rust_level_3: 21–50 %
- rust_level_4: más de 50 %

Reglas:
- Ignora daño mecánico, mordeduras, cicatrices o fondo; solo estima área de roya (manchas amarillo-naranja, pústulas, clorosis asociada a roya, necrosis ligada a esas lesiones).
- Elige SOLO una de las dos etiquetas en disputa del caso.
- No inventes otras enfermedades.
- Responde en español, breve y trazable.

Disputa por fila (izquierda / derecha según orden alfabético de File en cada par):
1: C2P1H1.jpg = rust_level_3 · C3P2H2.jpg = rust_level_4
2: C10P23H2.jpg = rust_level_2 · C12P32H2.jpg = rust_level_1
3: C11P12H2.jpg = rust_level_2 · C12P21H2.jpg = rust_level_1
4: C11P11E2.jpg = rust_level_4 · C12P19E1.jpg = rust_level_3
5: C11P11HE.jpg = rust_level_4 · C12P19H1.jpg = rust_level_3
6: C11P12E2.jpg = rust_level_2 · C12P21E2.jpg = rust_level_1
7: C11P18E2.jpg = rust_level_3 · C12P28E2.jpg = rust_level_2
8: C11P14E2.jpg = rust_level_2 · C11P26E2.jpg = rust_level_1
9: C11P18H2.jpg = rust_level_2 · C12P28H2.jpg = rust_level_1
10: C11P14H2.jpg = rust_level_2 · C11P26H2.jpg = rust_level_1
11: C11P15E2.jpg = rust_level_2 · C12P25E2.jpg = rust_level_1
12: C11P23H2.jpg = rust_level_3 · C12P34H2.jpg = rust_level_2
13: C11P21E2.jpg = rust_level_4 · C12P31E2.jpg = rust_level_3
14: C11P25E2.jpg = rust_level_2 · C12P38E2.jpg = rust_level_1
15: C11P25H2.jpg = rust_level_2 · C12P38H2.jpg = rust_level_1

Para cada caso 1–15 responde con este bloque:

Caso: <n>
Etiquetas en disputa: <label_a> vs <label_b>
Observación: <2–4 frases: qué se ve y % aproximado>
Chosen_label: <una de las dos>
Confidence: alta | media | baja
Kept_file: <archivo cuya etiqueta ORIGINAL ya coincide con chosen_label>
Withdrawn_file: <el otro archivo>
Justificación corta: <una frase>

Al final, una tabla CSV con columnas:
Case,chosen_label,kept_file,withdrawn_file,confidence,observation
```


Astra ya respondió con las 15 decisiones. Las leemos desde curated (`astra_rust_severity_decisions.csv`), sin volver a llamar al modelo.


In [ ]:
# CSV de decisiones de Astra (Claude Opus), ya guardado en curated.

astra_rust_severity_csv_path = Path(
    "../../data/curated/rocole/withdrawals/astra_rust_severity_decisions.csv"
)
astra_rust_severity_decisions = pd.read_csv(astra_rust_severity_csv_path)

astra_rust_severity_decisions[["case", "observation"]]


Completamos la tabla de revisión con lo de Astra: método generative_ai y modelo astra_claude_opus.


In [ ]:
# rust_severity_review_decisions, astra_rust_severity_decisions
# Si la tabla vacía no está en memoria, la armamos desde rust_severity_conflict_labels.

if "rust_severity_review_decisions" not in dir():
    decision_rows = []
    for case_number, group_number in enumerate(
        sorted(rust_severity_conflict_labels["group"].unique()), start=1
    ):
        group = (
            rust_severity_conflict_labels[
                rust_severity_conflict_labels["group"] == group_number
            ]
            .sort_values("File")
            .reset_index(drop=True)
        )
        decision_rows.append(
            {
                "case": case_number,
                "group": group_number,
                "file_a": group.loc[0, "File"],
                "label_a": group.loc[0, "Multiclass.Label"],
                "file_b": group.loc[1, "File"],
                "label_b": group.loc[1, "Multiclass.Label"],
                "chosen_label": pd.NA,
                "kept_file": pd.NA,
                "withdrawn_file": pd.NA,
                "decision_method": "generative_ai",
                "notes": pd.NA,
            }
        )
    rust_severity_review_decisions = pd.DataFrame(decision_rows)

astra_columns = astra_rust_severity_decisions[
    ["case", "chosen_label", "kept_file", "withdrawn_file", "confidence", "observation"]
]
rust_severity_review_decisions = rust_severity_review_decisions.drop(
    columns=["chosen_label", "kept_file", "withdrawn_file", "notes"],
    errors="ignore",
).merge(astra_columns, on="case", how="left")
rust_severity_review_decisions["decision_method"] = "generative_ai"
rust_severity_review_decisions["model"] = "astra_claude_opus"
rust_severity_review_decisions["notes"] = rust_severity_review_decisions["observation"]

rust_severity_review_decisions


Antes de guardar, comprobamos que hay 15 casos, que kept y withdrawn son el par, y que la etiqueta original de kept coincide con chosen_label.


In [ ]:
# rust_severity_review_decisions, rust_severity_conflict_labels

label_lookup = (
    rust_severity_conflict_labels.set_index("File")["Multiclass.Label"].to_dict()
)
check_rows = []
for _, row in rust_severity_review_decisions.iterrows():
    pair_files = {row["file_a"], row["file_b"]}
    check_rows.append(
        {
            "case": row["case"],
            "n_cases_ok": len(rust_severity_review_decisions) == 15,
            "pair_files_match": set([row["kept_file"], row["withdrawn_file"]])
            == pair_files,
            "kept_label_matches_chosen": label_lookup[row["kept_file"]]
            == row["chosen_label"],
        }
    )
rust_severity_decision_checks = pd.DataFrame(check_rows)

rust_severity_decision_checks


Guardamos el CSV de decisiones de severidad de roya en curated, con método generative_ai y modelo Astra Claude Opus.


In [ ]:
# rust_severity_review_decisions

rust_severity_decisions_path = Path(
    "../../data/curated/rocole/withdrawals/rust_severity_duplicate_decisions.csv"
)
rust_severity_decisions_path.parent.mkdir(parents=True, exist_ok=True)
rust_severity_review_decisions.to_csv(rust_severity_decisions_path, index=False)

rust_severity_decisions_path


De esas decisiones sacamos la lista de archivos que se retiran del trabajo.


In [ ]:
# rust_severity_review_decisions

rust_severity_withdrawals = rust_severity_review_decisions[
    ["case", "group", "withdrawn_file", "kept_file", "chosen_label"]
].copy()

rust_severity_withdrawals


Partimos de la tabla sin los 40 duplicados de misma etiqueta y quitamos estos 15 archivos retirados.


In [ ]:
# rocole_classes_same_label_deduped, rust_severity_withdrawals

withdrawn_files = set(rust_severity_withdrawals["withdrawn_file"])
rocole_classes_rust_severity_deduped = rocole_classes_same_label_deduped[
    ~rocole_classes_same_label_deduped["File"].isin(withdrawn_files)
].copy()

rocole_classes_rust_severity_deduped.shape


Actualizamos en curated la tabla de etiquetas de trabajo, ya con estos 15 retiros aplicados.


In [ ]:
# rocole_classes_rust_severity_deduped

rocole_classes_curated_path = Path("../../data/curated/rocole/classes.csv")
rocole_classes_curated_path.parent.mkdir(parents=True, exist_ok=True)
rocole_classes_rust_severity_deduped.to_csv(rocole_classes_curated_path, index=False)

rocole_classes_curated_path


Con esto quedan resueltos los 15 pares de solo severidad de roya (Astra). Siguen pendientes los 11 ácaro/roya y el par sana/ácaro.


Hay un par de fotos iguales con etiqueta binaria distinta (sana frente a ácaro). Filtramos los grupos donde Binary.Label no coincide.


In [ ]:
# duplicate_photo_labels

binary_conflict_group_numbers = []
for group_number, group in duplicate_photo_labels.groupby("group"):
    if group["Binary.Label"].nunique() > 1:
        binary_conflict_group_numbers.append(group_number)

binary_label_conflict_labels = duplicate_photo_labels[
    duplicate_photo_labels["group"].isin(binary_conflict_group_numbers)
].copy()

binary_label_conflict_labels


Contamos cuántos grupos son: debería ser uno solo.


In [ ]:
# binary_label_conflict_labels

binary_label_conflict_group_count = binary_label_conflict_labels["group"].nunique()

binary_label_conflict_group_count


Mostramos una sola foto del par (la primera por nombre). En el título van ambos archivos y ambas etiquetas binaria y multiclase. No decidimos todavía.


In [ ]:
# binary_label_conflict_labels, rocole_photos_dir, Image


group_number = sorted(binary_label_conflict_labels["group"].unique())[0]
pair_rows = (
    binary_label_conflict_labels[
        binary_label_conflict_labels["group"] == group_number
    ]
    .sort_values("File")
    .reset_index(drop=True)
)
first_row = pair_rows.iloc[0]
photo_path = rocole_photos_dir / first_row["File"]

title_parts = []
for _, row in pair_rows.iterrows():
    title_parts.append(
        f"{row['File']}: {row['Binary.Label']} / {row['Multiclass.Label']}"
    )
title = " | ".join(title_parts)

fig, ax = plt.subplots(figsize=(8, 6))
with Image.open(photo_path) as photo:
    ax.imshow(photo.convert("RGB"))
ax.set_xticks([])
ax.set_yticks([])
ax.set_title(title, fontsize=10)
fig.tight_layout()
plt.show()

# Una foto del par; no se escribe PDF.


Armamos la lista de retiro: las dos fotos del par salen del trabajo, con el mismo motivo de hold-out.


In [ ]:
# binary_label_conflict_labels

binary_label_conflict_withdrawals = binary_label_conflict_labels[
    ["group", "File", "Binary.Label", "Multiclass.Label"]
].copy()
binary_label_conflict_withdrawals["reason"] = (
    "binary_conflict_healthy_vs_mite_hold_out"
)

binary_label_conflict_withdrawals


Guardamos ese retiro en curated, aparte de la tabla de trabajo.


In [ ]:
# binary_label_conflict_withdrawals

binary_label_conflict_withdrawals_path = Path(
    "../../data/curated/rocole/withdrawals/binary_conflict_duplicate_withdrawals.csv"
)
binary_label_conflict_withdrawals_path.parent.mkdir(parents=True, exist_ok=True)
binary_label_conflict_withdrawals.to_csv(
    binary_label_conflict_withdrawals_path, index=False
)

binary_label_conflict_withdrawals_path


Quitamos ambas fotos de la tabla de trabajo. Si ya está en memoria la versión tras severidad de roya, la usamos; si no, leemos curated.


In [ ]:
# binary_label_conflict_withdrawals
# rocole_classes_rust_severity_deduped o curated

if "rocole_classes_rust_severity_deduped" in dir():
    working_table = rocole_classes_rust_severity_deduped
else:
    working_table = pd.read_csv(Path("../../data/curated/rocole/classes.csv"))

withdrawn_files = set(binary_label_conflict_withdrawals["File"])
rocole_classes_binary_conflict_deduped = working_table[
    ~working_table["File"].isin(withdrawn_files)
].copy()

rocole_classes_binary_conflict_deduped.shape


Sobreescribimos la tabla de etiquetas en curated con este retiro ya aplicado.


In [ ]:
# rocole_classes_binary_conflict_deduped

rocole_classes_curated_path = Path("../../data/curated/rocole/classes.csv")
rocole_classes_curated_path.parent.mkdir(parents=True, exist_ok=True)
rocole_classes_binary_conflict_deduped.to_csv(
    rocole_classes_curated_path, index=False
)

rocole_classes_curated_path


Las dos fotos del conflicto sana/ácaro quedan apartadas; el raw no se tocó.


Cerramos la primera parte: todo RoCoLe curado vive bajo `data/curated/rocole/`. Antes de armar las fotos, apartamos los 11 pares ácaro/roya (ambas copias) para que no queden dos nombres del mismo jpg.


Filtramos los grupos de fotos idénticas donde una etiqueta es ácaro y la otra es un nivel de roya.


In [ ]:
# duplicate_photo_labels

mite_rust_conflict_group_numbers = []
for group_number, group in duplicate_photo_labels.groupby("group"):
    multiclass_labels = set(group["Multiclass.Label"])
    has_mite = "red_spider_mite" in multiclass_labels
    has_rust = any(str(label).startswith("rust_level_") for label in multiclass_labels)
    same_binary = group["Binary.Label"].nunique() == 1
    if has_mite and has_rust and same_binary:
        mite_rust_conflict_group_numbers.append(group_number)

mite_rust_conflict_labels = duplicate_photo_labels[
    duplicate_photo_labels["group"].isin(mite_rust_conflict_group_numbers)
].copy()

mite_rust_conflict_labels


Contamos cuántos grupos son: deberían ser 11.


In [ ]:
# mite_rust_conflict_labels

mite_rust_conflict_group_count = mite_rust_conflict_labels["group"].nunique()

mite_rust_conflict_group_count


Apartamos ambas copias de cada par, con motivo de hold-out hasta revisar el diagnóstico.


In [ ]:
# mite_rust_conflict_labels

mite_rust_conflict_holdout = mite_rust_conflict_labels.copy()
mite_rust_conflict_holdout["reason"] = "mite_vs_rust_conflict_hold_out"

mite_rust_conflict_holdout


Guardamos ese hold-out en curated/rocole/withdrawals/.


In [ ]:
# mite_rust_conflict_holdout

mite_rust_conflict_holdout_path = Path(
    "../../data/curated/rocole/withdrawals/mite_rust_conflict_holdout.csv"
)
mite_rust_conflict_holdout_path.parent.mkdir(parents=True, exist_ok=True)
mite_rust_conflict_holdout.to_csv(mite_rust_conflict_holdout_path, index=False)

mite_rust_conflict_holdout_path


Quitamos esas 22 filas de la tabla de trabajo y dejamos classes.csv final de esta etapa.


In [ ]:
# tabla de trabajo tras el hold-out binario, y mite_rust_conflict_holdout

if "rocole_classes_binary_conflict_deduped" in dir():
    working_table = rocole_classes_binary_conflict_deduped
else:
    working_table = pd.read_csv(Path("../../data/curated/rocole/classes.csv"))

holdout_files = set(mite_rust_conflict_holdout["File"])
rocole_classes_curated = working_table[
    ~working_table["File"].isin(holdout_files)
].copy()

rocole_classes_curated_path = Path("../../data/curated/rocole/classes.csv")
rocole_classes_curated_path.parent.mkdir(parents=True, exist_ok=True)
rocole_classes_curated.to_csv(rocole_classes_curated_path, index=False)

rocole_classes_curated_path, rocole_classes_curated.shape


Materializamos en curated/rocole/photos/ solo las fotos de esa tabla (hardlink; si falla, copia). El raw no se borra.


In [ ]:
# rocole_classes_curated, rocole_photos_dir


curated_photos_dir = Path("../../data/curated/rocole/photos")
curated_photos_dir.mkdir(parents=True, exist_ok=True)

curated_photo_files = []
for file_name in rocole_classes_curated["File"]:
    source_path = rocole_photos_dir / file_name
    destination_path = curated_photos_dir / file_name
    if not destination_path.exists():
        try:
            destination_path.hardlink_to(source_path)
        except OSError:
            shutil.copy2(source_path, destination_path)
    curated_photo_files.append(destination_path.name)

len(curated_photo_files), curated_photos_dir


Comprobamos que photos/ y classes.csv digan lo mismo: misma cantidad y sin nombres de más.


In [ ]:
# rocole_classes_curated, curated_photos_dir

table_names = set(rocole_classes_curated["File"])
disk_names = {path.name for path in curated_photos_dir.glob("*.jpg")}
curated_photos_check = {
    "table_rows": len(table_names),
    "photos_on_disk": len(disk_names),
    "missing_on_disk": sorted(table_names - disk_names),
    "extra_on_disk": sorted(disk_names - table_names),
}

curated_photos_check


Primera parte del EDA de RoCoLe cerrada: etiquetas y fotos curadas en `data/curated/rocole/`. Los 11 pares ácaro/roya quedan en hold-out para otra revisión. Lo que sigue es características y el split en featured.


### Experimento aparte: patrón de nombres tipo C11P12

Apartamos un rato el EDA principal. Aquí miramos solo cómo se llaman las fotos en el Excel y en las 1560 fotos completas de raw: el patrón tipo `C11P12H1.jpg`. No reparte train/test; solo aclara si `C##P##` apunta a una planta y qué papel juegan H/E y el 1/2.


Sacamos de cada nombre de archivo la clave de planta (`C##P##`) y la cara (`H` o `E`). Trabajamos sobre las 1560 filas del Excel raw.


In [ ]:
# rocole_classes_path — Excel raw con las 1560 fotos


naming_experiment_table = pd.read_excel(rocole_classes_path)

plant_keys = []
leaf_faces = []
for file_name in naming_experiment_table["File"]:
    plant_match = re.match(r"^(C\d+)(P\d+)", file_name)
    face_match = re.search(r"(HE|[EH])\d+\.jpg$", file_name)
    plant_keys.append(
        plant_match.group(1) + plant_match.group(2) if plant_match else None
    )
    leaf_faces.append(face_match.group(1) if face_match else None)

naming_experiment_leaves = naming_experiment_table[
    ["File", "Binary.Label", "Multiclass.Label"]
].copy()
naming_experiment_leaves["plant_key"] = plant_keys
naming_experiment_leaves["leaf_face"] = leaf_faces

naming_experiment_leaves


Contamos cuántas fotos caen bajo cada `plant_key`. Queremos ver si el papel de 390 plantas × 4 fotos se sostiene en estos nombres.


In [ ]:
# naming_experiment_leaves

plant_group_sizes = naming_experiment_leaves.groupby("plant_key").size()
n_plants = int(plant_group_sizes.shape[0])
all_plants_have_four_photos = bool((plant_group_sizes == 4).all())

plant_group_sizes, n_plants, all_plants_have_four_photos


Tomamos un ejemplo concreto: la planta `C11P12`, con sus cuatro archivos y etiquetas.


In [ ]:
# naming_experiment_leaves

plant_c11p12_rows = naming_experiment_leaves[
    naming_experiment_leaves["plant_key"] == "C11P12"
].copy()

plant_c11p12_rows


Cómo evaluamos el patrón en este ejemplo: (1) el conteo estructural 390 × 4 frente a lo que dice el paper; (2) las etiquetas del cuarteto `C11P12`; (3) una mirada visual del mosaico 2×2. Abajo van las cuatro fotos de raw, con el nombre y la etiqueta debajo de cada una.


In [ ]:
# plant_c11p12_rows, rocole_photos_dir, Image


mosaic_rows = plant_c11p12_rows.sort_values("File").reset_index(drop=True)
figure, axes = plt.subplots(2, 2, figsize=(10, 10))

for axis, (_, row) in zip(axes.flat, mosaic_rows.iterrows()):
    photo_path = rocole_photos_dir / row["File"]
    with Image.open(photo_path) as photo:
        axis.imshow(photo.convert("RGB"))
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_xlabel(
        f"{row['File']}\n{row['Binary.Label']} | {row['Multiclass.Label']}",
        fontsize=9,
    )

figure.suptitle("Planta C11P12 — cuatro fotos (cara × 1/2)", fontsize=13)
figure.tight_layout()

results_dir = Path("../../experiments/results")
results_dir.mkdir(parents=True, exist_ok=True)
plant_c11p12_mosaic_path = results_dir / "rocole_plant_c11p12_mosaic.png"
figure.savefig(plant_c11p12_mosaic_path, dpi=120, bbox_inches="tight")
plt.show()

plant_c11p12_mosaic_path


Cruzamos esas cuatro fotos con `data/curated/rocole/classes.csv` para ver cuáles siguen en curated después de limpiezas y hold-outs.


In [ ]:
# plant_c11p12_rows

curated_classes = pd.read_csv(Path("../../data/curated/rocole/classes.csv"))
plant_c11p12_in_curated = plant_c11p12_rows.copy()
plant_c11p12_in_curated["in_curated"] = plant_c11p12_in_curated["File"].isin(
    set(curated_classes["File"])
)

plant_c11p12_in_curated


Tras `C11P12`, miramos dos plantas más (`C1P1` y `C10P23`) para ver si el mismo patrón de nombres se repite.


Filtramos las cuatro filas de la planta `C1P1`.


In [ ]:
# naming_experiment_leaves

plant_c1p1_rows = naming_experiment_leaves[
    naming_experiment_leaves["plant_key"] == "C1P1"
].copy()

plant_c1p1_rows


Armamos el mosaico 2×2 de `C1P1` con las cuatro fotos de raw, nombre y etiqueta debajo de cada una.


In [ ]:
# plant_c1p1_rows, rocole_photos_dir, Image


mosaic_rows = plant_c1p1_rows.sort_values("File").reset_index(drop=True)
figure, axes = plt.subplots(2, 2, figsize=(10, 10))

for axis, (_, row) in zip(axes.flat, mosaic_rows.iterrows()):
    photo_path = rocole_photos_dir / row["File"]
    with Image.open(photo_path) as photo:
        axis.imshow(photo.convert("RGB"))
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_xlabel(
        f"{row['File']}\n{row['Binary.Label']} | {row['Multiclass.Label']}",
        fontsize=9,
    )

figure.suptitle("Planta C1P1 — cuatro fotos (cara × 1/2)", fontsize=13)
figure.tight_layout()

results_dir = Path("../../experiments/results")
results_dir.mkdir(parents=True, exist_ok=True)
plant_c1p1_mosaic_path = results_dir / "rocole_plant_c1p1_mosaic.png"
figure.savefig(plant_c1p1_mosaic_path, dpi=120, bbox_inches="tight")
plt.show()

plant_c1p1_mosaic_path


Filtramos las cuatro filas de la planta `C10P23`.


In [ ]:
# naming_experiment_leaves

plant_c10p23_rows = naming_experiment_leaves[
    naming_experiment_leaves["plant_key"] == "C10P23"
].copy()

plant_c10p23_rows


Armamos el mosaico 2×2 de `C10P23` con las cuatro fotos de raw, nombre y etiqueta debajo de cada una.


In [ ]:
# plant_c10p23_rows, rocole_photos_dir, Image


mosaic_rows = plant_c10p23_rows.sort_values("File").reset_index(drop=True)
figure, axes = plt.subplots(2, 2, figsize=(10, 10))

for axis, (_, row) in zip(axes.flat, mosaic_rows.iterrows()):
    photo_path = rocole_photos_dir / row["File"]
    with Image.open(photo_path) as photo:
        axis.imshow(photo.convert("RGB"))
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_xlabel(
        f"{row['File']}\n{row['Binary.Label']} | {row['Multiclass.Label']}",
        fontsize=9,
    )

figure.suptitle("Planta C10P23 — cuatro fotos (cara × 1/2)", fontsize=13)
figure.tight_layout()

results_dir = Path("../../experiments/results")
results_dir.mkdir(parents=True, exist_ok=True)
plant_c10p23_mosaic_path = results_dir / "rocole_plant_c10p23_mosaic.png"
figure.savefig(plant_c10p23_mosaic_path, dpi=120, bbox_inches="tight")
plt.show()

plant_c10p23_mosaic_path


Conclusión de este bloque. En los grupos revisados (`C11P12`, `C1P1`, `C10P23`) la lectura del nombre se sostiene a ojo: `C##P##` agrupa la planta, `H`/`E` la cara y el `1`/`2` suele separar una hoja sana de una afectada. Eso lo validamos en estos ejemplos; el artículo de RoCoLe no documenta esa nomenclatura de forma explícita, aunque sí dice que hay cuatro fotos por planta.

Antes de partir train/test conviene agrupar por `plant_key` y, además, mirar los vínculos entre prefijos que ya vimos por duplicados exactos (misma imagen con otro nombre). Si no, grupos relacionados pueden caer en particiones distintas y filtrar información.


Dejamos en curated la columna `plant_key` (y la cara de la hoja). Los vínculos entre plantas que salieron de los duplicados se usarán al partir train/test; aquí solo guardamos la columna en la tabla.


Aseguramos la tabla curated en memoria: si ya la tenemos la reutilizamos; si no, la leemos de `../../data/curated/rocole/classes.csv`.


In [ ]:
# rocole_classes_curated (si está en memoria) o ../../data/curated/rocole/classes.csv

rocole_classes_curated_path = Path("../../data/curated/rocole/classes.csv")
try:
    rocole_classes_curated = rocole_classes_curated.copy()
except NameError:
    rocole_classes_curated = pd.read_csv(rocole_classes_curated_path)

rocole_classes_curated.head(), rocole_classes_curated.shape


Sacamos `plant_key` del nombre del archivo con el mismo patrón del experimento: el prefijo `C##P##`.


In [ ]:
rocole_classes_curated = rocole_classes_curated.copy()


plant_keys = []
for file_name in rocole_classes_curated["File"]:
    plant_match = re.match(r"^(C\d+P\d+)", str(file_name))
    plant_keys.append(plant_match.group(1) if plant_match else None)
rocole_classes_curated["plant_key"] = plant_keys

rocole_classes_curated[["File", "plant_key"]].head()


Después del prefijo de planta, leemos la cara de la hoja: `H`, `E` o `HE`.


In [ ]:
rocole_classes_curated = rocole_classes_curated.copy()

leaf_faces = []
for file_name, plant_key in zip(
    rocole_classes_curated["File"], rocole_classes_curated["plant_key"]
):
    file_text = str(file_name)
    if plant_key and file_text.startswith(str(plant_key)):
        after_plant = file_text[len(str(plant_key)) :]
    else:
        after_plant = file_text
    face_match = re.match(r"^(HE|[EH])", after_plant)
    leaf_faces.append(face_match.group(1) if face_match else None)
rocole_classes_curated["leaf_face"] = leaf_faces

rocole_classes_curated[["File", "plant_key", "leaf_face"]].head()


Reescribimos `../../data/curated/rocole/classes.csv` con `plant_key` y `leaf_face`.


In [ ]:
# rocole_classes_curated

rocole_classes_curated_path = Path("../../data/curated/rocole/classes.csv")
rocole_classes_curated_path.parent.mkdir(parents=True, exist_ok=True)
rocole_classes_curated.to_csv(rocole_classes_curated_path, index=False)

rocole_classes_curated_path, rocole_classes_curated.shape, list(
    rocole_classes_curated.columns
)


Curated ya lleva `plant_key` y `leaf_face` por foto. Al partir train/test conviene agrupar por planta y, aparte, mirar los vínculos de duplicados.


Repartimos las fotos curadas en entrenamiento, validación y prueba. El corte va por grupos de planta vinculados (no foto a foto), para que la misma planta —ni un duplicado con otro nombre— no filtre a dos lados.


Partimos de la tabla ya curada: 1481 fotos con planta y cara.


In [ ]:
# ../../data/curated/rocole/classes.csv

leaves = pd.read_csv("../../data/curated/rocole/classes.csv")

leaves.head(), leaves.shape


Los retiros de duplicados con la misma etiqueta nos dicen qué plantas estaban ligadas aunque el nombre `C##P##` cambie.


In [ ]:
# ../../data/curated/rocole/withdrawals/same_label_duplicate_withdrawals.csv

same_label_withdrawals = pd.read_csv(
    "../../data/curated/rocole/withdrawals/same_label_duplicate_withdrawals.csv"
)

same_label_withdrawals.head(), same_label_withdrawals.shape


Lo mismo con las decisiones de severidad de roya: kept y withdrawn pueden ser plantas distintas.


In [ ]:
# ../../data/curated/rocole/withdrawals/rust_severity_duplicate_decisions.csv

rust_severity_withdrawals = pd.read_csv(
    "../../data/curated/rocole/withdrawals/rust_severity_duplicate_decisions.csv"
)

rust_severity_withdrawals.head(), rust_severity_withdrawals.shape


De cada par kept/withdrawn de same_label sacamos el `plant_key` de ambos archivos.


In [ ]:
same_label_withdrawals = same_label_withdrawals.copy()


def plant_key_from_file(file_name):
    plant_match = re.match(r"^(C\d+P\d+)", str(file_name))
    return plant_match.group(1) if plant_match else None

same_label_plant_edges = []
for kept_file, withdrawn_file in zip(
    same_label_withdrawals["kept_file"],
    same_label_withdrawals["withdrawn_file"],
):
    plant_a = plant_key_from_file(kept_file)
    plant_b = plant_key_from_file(withdrawn_file)
    if plant_a and plant_b and plant_a != plant_b:
        same_label_plant_edges.append((plant_a, plant_b))

len(same_label_plant_edges), same_label_plant_edges[:5]


Repetimos con las decisiones de severidad de roya.


In [ ]:
rust_severity_withdrawals = rust_severity_withdrawals.copy()

rust_severity_plant_edges = []
for kept_file, withdrawn_file in zip(
    rust_severity_withdrawals["kept_file"],
    rust_severity_withdrawals["withdrawn_file"],
):
    plant_a = plant_key_from_file(kept_file)
    plant_b = plant_key_from_file(withdrawn_file)
    if plant_a and plant_b and plant_a != plant_b:
        rust_severity_plant_edges.append((plant_a, plant_b))

len(rust_severity_plant_edges), rust_severity_plant_edges[:5]


Juntamos todas las aristas: si A se duplicó con B, A y B son el mismo grupo vinculado.


In [ ]:
# same_label_plant_edges, rust_severity_plant_edges

plant_link_edges = list(same_label_plant_edges) + list(rust_severity_plant_edges)

len(plant_link_edges), plant_link_edges[:5]


Cada planta queda en un componente: sola o unida a las que vinieron de duplicados.


In [ ]:
# plant_link_edges, leaves

parent = {}

def find(plant):
    parent.setdefault(plant, plant)
    while parent[plant] != plant:
        parent[plant] = parent[parent[plant]]
        plant = parent[plant]
    return plant

def union(plant_a, plant_b):
    root_a = find(plant_a)
    root_b = find(plant_b)
    if root_a != root_b:
        parent[root_b] = root_a

for plant_key in leaves["plant_key"].dropna().unique():
    find(str(plant_key))

for plant_a, plant_b in plant_link_edges:
    union(str(plant_a), str(plant_b))

root_to_group = {}
next_group_id = 0
plant_to_link_group = {}
for plant_key in sorted(leaves["plant_key"].dropna().astype(str).unique()):
    root = find(plant_key)
    if root not in root_to_group:
        root_to_group[root] = next_group_id
        next_group_id += 1
    plant_to_link_group[plant_key] = root_to_group[root]

len(plant_to_link_group), next_group_id, list(plant_to_link_group.items())[:5]


Pegamos el grupo vinculado a cada foto.


In [ ]:
leaves = leaves.copy()

leaves["link_group"] = leaves["plant_key"].astype(str).map(plant_to_link_group)

leaves[["File", "plant_key", "link_group"]].head(), leaves["link_group"].isna().sum()


Para estratificar sin mirar foto a foto, cada grupo lleva una etiqueta binaria de mayoría (sana vs afectada). Si empatan, contamos el grupo como afectada.


In [ ]:
leaves = leaves.copy()

stratum_rows = []
for link_group, group_leaves in leaves.groupby("link_group"):
    n_healthy = (group_leaves["Binary.Label"] == "healthy").sum()
    n_unhealthy = (group_leaves["Binary.Label"] == "unhealthy").sum()
    # empate → unhealthy
    stratum = "healthy" if n_healthy > n_unhealthy else "unhealthy"
    stratum_rows.append({"link_group": link_group, "stratum": stratum})

link_group_stratum = pd.DataFrame(stratum_rows)

link_group_stratum.head(), link_group_stratum["stratum"].value_counts()


Fijamos la semilla para que el reparto sea reproducible en el equipo.


In [ ]:
# Semilla fija del proyecto: mismo corte en todos los experimentos.
seed = 42

seed


Repartimos los grupos (no las fotos sueltas) en 70 % entrenamiento, 15 % validación y 15 % prueba, respetando el estrato.


In [ ]:
# link_group_stratum, seed

try:
except ImportError:
    train_test_split = None

groups = link_group_stratum.copy()

if train_test_split is not None:
    train_groups, temp_groups = train_test_split(
        groups,
        test_size=0.30,
        random_state=seed,
        stratify=groups["stratum"],
    )
    val_groups, test_groups = train_test_split(
        temp_groups,
        test_size=0.50,
        random_state=seed,
        stratify=temp_groups["stratum"],
    )
else:

    rng = np.random.default_rng(seed)
    assigned = []
    for stratum_name, stratum_groups in groups.groupby("stratum"):
        ids = stratum_groups["link_group"].to_numpy().copy()
        rng.shuffle(ids)
        n = len(ids)
        n_train = int(round(n * 0.70))
        n_val = int(round(n * 0.15))
        # el resto va a test (~15 %)
        for link_group in ids[:n_train]:
            assigned.append((link_group, "train"))
        for link_group in ids[n_train : n_train + n_val]:
            assigned.append((link_group, "val"))
        for link_group in ids[n_train + n_val :]:
            assigned.append((link_group, "test"))
    link_group_split = pd.DataFrame(assigned, columns=["link_group", "split"])
    train_groups = test_groups = val_groups = None

if train_test_split is not None:
    link_group_split = pd.concat(
        [
            train_groups.assign(split="train")[["link_group", "split"]],
            val_groups.assign(split="val")[["link_group", "split"]],
            test_groups.assign(split="test")[["link_group", "split"]],
        ],
        ignore_index=True,
    )

link_group_split["split"].value_counts(), link_group_split.head()


Cada foto hereda el split de su grupo vinculado.


In [ ]:
leaves = leaves.copy()

split_by_group = dict(
    zip(link_group_split["link_group"], link_group_split["split"])
)
leaves["split"] = leaves["link_group"].map(split_by_group)

leaves[["File", "link_group", "split"]].head(), leaves["split"].value_counts()


Comprobamos que ningún `link_group` aparezca en más de una partición.


In [ ]:
leaves = leaves.copy()

link_group_split_check = (
    leaves.groupby("link_group")["split"].nunique().reset_index(name="n_splits")
)
groups_in_multiple_splits = link_group_split_check[
    link_group_split_check["n_splits"] > 1
]

len(groups_in_multiple_splits), groups_in_multiple_splits.head()


Miramos cuántas fotos sanas y afectadas quedaron en cada partición.


In [ ]:
leaves = leaves.copy()

binary_split_counts = (
    leaves.groupby(["split", "Binary.Label"]).size().unstack(fill_value=0)
)

binary_split_counts


Miramos también el desglose multiclass por partición (solo para revisar, no para estratificar).


In [ ]:
leaves = leaves.copy()

multiclass_split_counts = (
    leaves.groupby(["split", "Multiclass.Label"]).size().unstack(fill_value=0)
)

multiclass_split_counts


Armamos la ruta de la foto respecto a las photos curadas y dejamos solo las columnas del índice.


In [ ]:
leaves = leaves.copy()

rocole_index = leaves[
    [
        "File",
        "plant_key",
        "link_group",
        "Binary.Label",
        "Multiclass.Label",
        "leaf_face",
        "split",
    ]
].copy()
rocole_index["photo_path"] = rocole_index["File"].map(
    lambda file_name: f"data/curated/rocole/photos/{file_name}"
)

rocole_index.head(), rocole_index.shape


Guardamos el índice en featured, alineado a curated por dataset: `data/featured/rocole/index.csv`.


In [ ]:
# rocole_index

rocole_index_path = Path("../../data/featured/rocole/index.csv")
rocole_index_path.parent.mkdir(parents=True, exist_ok=True)
rocole_index.to_csv(rocole_index_path, index=False)

rocole_index_path, rocole_index.shape, list(rocole_index.columns)


Curated ya no se toca: el split vive en `data/featured/rocole/index.csv`. Ningún grupo vinculado cruza particiones. Lo que sigue, en otro cuaderno, son características solo con las hojas de entrenamiento.
